In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyproj import Transformer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os
import sys

def process_city_data(city_name, input_filename=None, year=None):
    """
    Process building data for a specific city with optional year tagging.
    
    Args:
        city_name (str): Name of the city to process
        input_filename (str, optional): Custom input filename
        year (int, optional): Year to tag this dataset with
    """
    print(f"Processing data for {city_name}...")
    
    # Load data with flexible filename
    if input_filename is None:
        input_file = f"{city_name.lower()}.csv"
    else:
        input_file = input_filename
        
    if not os.path.exists(input_file):
        print(f"Error: Input file {input_file} not found!")
        return
    
    df = pd.read_csv(input_file)
    
    # Convert coordinates
    def convert_coordinates(df):
        transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)
        lon, lat = transformer.transform(df["coordonnee_cartographique_x_ban"].values,
                                         df["coordonnee_cartographique_y_ban"].values)
        df["longitude"] = lon
        df["latitude"] = lat
        return df

    try:
        df = convert_coordinates(df)
    except KeyError as e:
        print(f"Warning: Coordinate conversion failed. Missing column: {e}")
        print("Continuing without coordinate conversion...")
    
    # Rename and engineer features
    df = df.rename(columns={
        "numero_dpe": "building_id",
        "conso_5 usages_ef": "Energy_Consumption",
        "emission_ges_5_usages": "CO2_Usage",
        "etiquette_dpe": "true_energy_label",
        "etiquette_ges": "true_ges_label"
    })
    
    # Check if required columns exist before calculations
    if "Energy_Consumption" in df.columns:
        df["Water_Usage"] = df["Energy_Consumption"] * 0.3
    else:
        print("Warning: Energy_Consumption column not found, skipping Water_Usage calculation")
        
    if "conso_5 usages_par_m2_ef" in df.columns:
        df["Energy_Intensity"] = df["conso_5 usages_par_m2_ef"]
    else:
        print("Warning: conso_5 usages_par_m2_ef column not found")
        
    if "emission_ges_5_usages par_m2" in df.columns:
        df["CO2_Intensity"] = df["emission_ges_5_usages par_m2"]
    else:
        print("Warning: emission_ges_5_usages par_m2 column not found")

    final_cols = [
        #–– Primary key & location ––
        "building_id",
        "latitude", "longitude",

        #–– Core performance metrics ––
        "Energy_Consumption", "CO2_Usage", "Water_Usage",
        "Energy_Intensity", "CO2_Intensity",

        #–– Official DPE labels ––
        "true_energy_label", "true_ges_label",

        #–– BAN address fields (normalized) ––
        "adresse_ban",         # full standardized address
        "numero_voie_ban",     # street number
        "nom_rue_ban",         # street name
        "code_postal_ban",     # postal code
        "nom_commune_ban",     # commune name
        "identifiant_ban",     # BAN address ID

        #–– Raw-fallback address fields ––
        "adresse_brut",
        "nom_commune_brut",
        "code_postal_brut",

        #–– Basic building attributes for filters ––
        "annee_construction",
        "surface_habitable_immeuble",
    ]

    # Filter only columns that exist in the dataframe
    available_cols = [col for col in final_cols if col in df.columns]
    print(f"Available columns: {len(available_cols)}/{len(final_cols)}")
    
    required = [
        "building_id", "Energy_Consumption", "CO2_Usage", "Water_Usage", 
        "Energy_Intensity", "CO2_Intensity"
    ]
    
    # Adjust required columns if location data is missing
    if "latitude" in df.columns and "longitude" in df.columns:
        required.extend(["latitude", "longitude"])
    
    # Check which required columns are available
    available_required = [col for col in required if col in df.columns]
    missing_required = [col for col in required if col not in df.columns]
    
    if missing_required:
        print(f"Warning: Missing required columns: {missing_required}")
    
    # select & only drop rows missing available required columns
    df_clean = df[available_cols].dropna(subset=available_required)
    
    # Add log-transformed features
    numeric_cols = [
        "Energy_Consumption", "CO2_Usage", 
        "Energy_Intensity", "CO2_Intensity"
    ]
    
    # Only transform columns that exist
    existing_numeric = [col for col in numeric_cols if col in df_clean.columns]
    for col in existing_numeric:
        df_clean[f"log1p_{col}"] = np.log1p(df_clean[col])
    
    # Add year tag if provided
    df_clean['city'] = city_name
    if year is not None:
        df_clean['year'] = year
        output_file = f"reduced_{city_name.lower()}_buildings_{year}.csv"
    else:
        output_file = f"reduced_{city_name.lower()}_buildings.csv"
    
    # Save cleaned dataset
    df_clean.to_csv(output_file, index=False)
    print(f"Clean dataset saved to {output_file}.")
    
    return df_clean

def draw_multipliers(n_buildings):
    """
    Generate multipliers for synthetic future data.
    
    Args:
        n_buildings (int): Number of buildings
    
    Returns:
        numpy.ndarray: Array of multipliers
    """
    # choose a distribution "scenario" for each building:
    # 60% get Normal noise, 30% Uniform, 10% Log-Normal "jumps"
    scenarios = np.random.choice(
        ["normal", "uniform", "lognormal"],
        size=n_buildings,
        p=[0.6, 0.3, 0.1]
    )
    m = np.zeros(n_buildings)
    for i, scenario in enumerate(scenarios):
        if scenario == "normal":
            m[i] = np.random.normal(loc=1.0, scale=0.05)
        elif scenario == "uniform":
            m[i] = np.random.uniform(0.9, 1.1)
        else:  # lognormal: small chance of a bigger jump
            m[i] = np.random.lognormal(mean=0, sigma=0.1)
    return m

def generate_future_data(df, city_name, target_year):
    """
    Generate synthetic data for future year based on existing data.
    
    Args:
        df (DataFrame): Source dataframe
        city_name (str): Name of the city
        target_year (int): Target year for synthetic data
    """
    print(f"Generating synthetic data for {city_name}, year {target_year}...")
    
    # Make a copy of the dataframe
    future_df = df.copy()
    future_df['city'] = city_name
    # Update year
    future_df['year'] = target_year
    
    # Get the number of buildings
    n_buildings = len(future_df)
    
    # Generate multipliers for different metrics
    energy_multipliers = draw_multipliers(n_buildings)
    co2_multipliers = draw_multipliers(n_buildings)
    water_multipliers = draw_multipliers(n_buildings)
    
    # Apply multipliers to core metrics
    if "Energy_Consumption" in future_df.columns:
        future_df["Energy_Consumption"] = future_df["Energy_Consumption"] * energy_multipliers
        
    if "Energy_Intensity" in future_df.columns:
        future_df["Energy_Intensity"] = future_df["Energy_Intensity"] * energy_multipliers
        
    if "CO2_Usage" in future_df.columns:
        future_df["CO2_Usage"] = future_df["CO2_Usage"] * co2_multipliers
        
    if "CO2_Intensity" in future_df.columns:
        future_df["CO2_Intensity"] = future_df["CO2_Intensity"] * co2_multipliers
        
    if "Water_Usage" in future_df.columns:
        future_df["Water_Usage"] = future_df["Water_Usage"] * water_multipliers
    
    # Re-calculate log transformations with new values
    numeric_cols = [
        "Energy_Consumption", "CO2_Usage", 
        "Energy_Intensity", "CO2_Intensity"
    ]
    
    # Only transform columns that exist
    existing_numeric = [col for col in numeric_cols if col in future_df.columns]
    for col in existing_numeric:
        log_col = f"log1p_{col}"
        if log_col in future_df.columns:  # Update existing log columns
            future_df[log_col] = np.log1p(future_df[col])
    
    # Generate some random changes in labels if they exist
    if "true_energy_label" in future_df.columns:
        # Randomly improve some labels (5% chance)
        mask = np.random.random(size=n_buildings) < 0.05
        label_map = {"G": "F", "F": "E", "E": "D", "D": "C", "C": "B", "B": "A", "A": "A"}
        future_df.loc[mask, "true_energy_label"] = future_df.loc[mask, "true_energy_label"].map(
            lambda x: label_map.get(x, x) if isinstance(x, str) else x
        )
    
    if "true_ges_label" in future_df.columns:
        # Randomly improve some labels (5% chance)
        mask = np.random.random(size=n_buildings) < 0.05
        future_df.loc[mask, "true_ges_label"] = future_df.loc[mask, "true_ges_label"].map(
            lambda x: label_map.get(x, x) if isinstance(x, str) else x
        )
    
    # Save future dataset
    output_file = f"reduced_{city_name.lower()}_buildings_{target_year}.csv"
    future_df.to_csv(output_file, index=False)
    print(f"Synthetic {target_year} data saved to {output_file}.")
    
    # Run K-Means clustering on the future data
    try:
        cluster_future_data(future_df, city_name, target_year)
    except Exception as e:
        print(f"Warning: Could not perform clustering on future data: {e}")
    
    return future_df

def cluster_future_data(df, city_name, year):
    """
    Apply K-means clustering to future data.
    
    Args:
        df (DataFrame): Data to cluster
        city_name (str): City name
        year (int): Year of data
    """
    # Find log columns for clustering
    log_cols = [col for col in df.columns if col.startswith('log1p_')]
    
    if len(log_cols) < 2:
        print("Not enough log-transformed columns for clustering")
        return df
    
    # Prepare data for clustering
    X = df[log_cols].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # Apply PCA
    pca = PCA(n_components=2)
    pcs = pca.fit_transform(X_scaled)
    df["PC1"], df["PC2"] = pcs[:,0], pcs[:,1]
    
    # Apply K-Means
    kmeans = KMeans(n_clusters=4, random_state=42)
    df["cluster"] = kmeans.fit_predict(X_scaled)
    
    
    
    return df

def analyze_city_data(df, city_name, year=None):
    """
    Analyze the processed city data and generate visualizations.
    
    Args:
        df (DataFrame): Processed dataframe
        city_name (str): City name for titles
        year (int, optional): Year of the data for titles
    """
    title_prefix = f"{city_name}"
    if year is not None:
        title_prefix = f"{city_name} ({year})"
    
    print(f"\nAnalyzing data for {title_prefix}...")
    
    # Quick overview
    print("===== DATA INFO =====")
    df.info()
    print("\n===== MISSING VALUES =====")
    print(df.isnull().sum())
    print("\n===== DESCRIPTIVE STATISTICS =====")
    print(df.describe().T)
    
    # Check for numeric columns availability
    base_numeric_cols = [
        "Energy_Consumption", "CO2_Usage", "Water_Usage",
        "Energy_Intensity", "CO2_Intensity"
    ]
    numeric_cols = [col for col in base_numeric_cols if col in df.columns]
    
    if len(numeric_cols) < 2:
        print("Insufficient numeric columns for analysis.")
        return
    
    # Univariate analysis
    for col in numeric_cols:
        plt.figure(figsize=(8,4))
        sns.histplot(df[col], kde=True)
        plt.title(f"{title_prefix} - {col} — Distribution + KDE")
        plt.tight_layout()
        plt.close()

        plt.figure(figsize=(6,2))
        sns.boxplot(x=df[col])
        plt.title(f"{title_prefix} - {col} — Boxplot")
        plt.tight_layout()
        plt.close()

    # Correlation matrix
    corr = df[numeric_cols].corr()
    print("\n===== CORRELATION MATRIX =====")
    print(corr)
    plt.figure(figsize=(6,5))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", square=True)
    plt.title(f"{title_prefix} - Correlation Matrix")
    plt.tight_layout()
    plt.close()

    # Pairwise scatterplots
    sns.pairplot(df[numeric_cols], diag_kind="kde", corner=True)
    plt.suptitle(f"{title_prefix} - Pairplot of Numeric Features", y=1.02)
    plt.tight_layout()
    plt.close()

    # Geographic scatter if coordinates are available
    if "longitude" in df.columns and "latitude" in df.columns:
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=df["longitude"], y=df["latitude"], 
                        hue=df["Energy_Consumption"] if "Energy_Consumption" in df.columns else None, 
                        palette="viridis", s=20)
        plt.title(f"{title_prefix} - Map Scatter: Energy Consumption")
        plt.tight_layout()
        plt.close()

        # Geo‑bubble plots
        for col in numeric_cols:
            plt.figure(figsize=(6,6))
            sns.scatterplot(x=df["longitude"], y=df["latitude"], 
                            size=df[col], sizes=(10,200), alpha=0.6, legend=False)
            plt.title(f"{title_prefix} - Geo‑Bubble: {col}")
            plt.tight_layout()
            plt.close()

    # Check for existing log columns
    log_cols = [col for col in df.columns if col.startswith('log1p_')]
    
    if log_cols:
        corr2 = df[log_cols].corr()
        plt.figure(figsize=(5,4))
        sns.heatmap(corr2, annot=True, fmt=".2f", cmap="coolwarm", square=True)
        plt.title(f"{title_prefix} - Corr of Log‑Transformed Features")
        plt.close()

        sns.pairplot(df[log_cols], diag_kind="kde", corner=True)
        plt.suptitle(f"{title_prefix} - Pairplot of Log‑Transformed Metrics", y=1.02)
        plt.tight_layout()
        plt.close()
    
    # PCA visualization if PC columns exist
    if "PC1" in df.columns and "PC2" in df.columns:
        plt.figure(figsize=(6,5))
        sns.scatterplot(x="PC1", y="PC2", data=df, alpha=0.3)
        plt.title(f"{title_prefix} - PCA of Building Metrics")
        plt.tight_layout()
        plt.close()
        
        # K-Means visualization if cluster column exists
        if "cluster" in df.columns:
            plt.figure(figsize=(6,5))
            sns.scatterplot(x="PC1", y="PC2", hue="cluster", data=df, palette="tab10", alpha=0.5)
            plt.title(f"{title_prefix} - K‑Means Clusters on PCA space")
            plt.tight_layout()
            plt.close()
    
    return df

def process_city_with_years(city_name, input_filename=None, base_year=2024, future_years=[2025]):
    """
    Process city data and generate data for multiple years.
    
    Args:
        city_name (str): Name of the city
        input_filename (str, optional): Input filename
        base_year (int): Base year for initial data
        future_years (list): List of future years to generate
    
    Returns:
        dict: Dictionary of dataframes by year
    """
    # Process base year data
    base_df = process_city_data(city_name, input_filename, base_year)
    if base_df is None:
        return None
        
    # Apply clustering to base data
    try:
        cluster_future_data(base_df, city_name, base_year)
    except Exception as e:
        print(f"Warning: Could not perform clustering on base data: {e}")
    
    # Store all dataframes in a dictionary
    all_dfs = {base_year: base_df}
    
    # Generate data for future years
    current_df = base_df
    for year in future_years:
        future_df = generate_future_data(current_df, city_name, year)
        all_dfs[year] = future_df
        # Use the latest year as base for the next year (cumulative changes)
        current_df = future_df
    
    # Create a combined dataset with all years
    combined_df = pd.concat(all_dfs.values())
    combined_output = f"reduced_{city_name.lower()}_buildings_all_years.csv"
    combined_df.to_csv(combined_output, index=False)
    print(f"Combined multi-year dataset saved to {combined_output}.")
    
    return all_dfs

In [5]:
city_dfs = process_city_with_years(
    "gordes", 
    input_filename="gordes.csv",  
    base_year=2024, 
    future_years=[2025]
)


Processing data for gordes...
Available columns: 21/21
Clean dataset saved to reduced_gordes_buildings_2024.csv.
Generating synthetic data for gordes, year 2025...
Synthetic 2025 data saved to reduced_gordes_buildings_2025.csv.
Combined multi-year dataset saved to reduced_gordes_buildings_all_years.csv.


c:\Users\USER\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
c:\Users\USER\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


In [7]:
city_dfs = process_city_with_years(
    "lyon", 
    input_filename="Lyon.csv",  
    base_year=2024, 
    future_years=[2025]
)


Processing data for lyon...
Continuing without coordinate conversion...
Available columns: 6/21
Clean dataset saved to reduced_lyon_buildings_2024.csv.
Generating synthetic data for lyon, year 2025...
Synthetic 2025 data saved to reduced_lyon_buildings_2025.csv.
Combined multi-year dataset saved to reduced_lyon_buildings_all_years.csv.


In [ ]:
def process_uploaded_data(uploaded_file, username=None):
    """
    Process an uploaded CSV file with flexible column handling.
    Uses the same processing logic as process_city_data to ensure consistency.
    
    Args:
        uploaded_file: File-like object from Streamlit file_uploader
        username: Username who uploaded the file
    
    Returns:
        tuple: (processed_dataframe, city_name)
    """
    import pandas as pd
    import numpy as np
    import uuid
    from datetime import datetime
    import os
    import traceback
    
    print(f"Processing uploaded file: {uploaded_file.name}")
    
    try:
        # Read CSV data
        df = pd.read_csv(uploaded_file)
        print(f"Loaded CSV with {len(df)} rows and {len(df.columns)} columns")
        print(f"Columns found: {df.columns.tolist()}")
        
        # Generate a city name based on the filename
        city_name = os.path.splitext(uploaded_file.name)[0]
        print(f"Using city name: {city_name}")
        
        # Create a copy to avoid modifying the original
        df_processed = df.copy()
        
        # Step 1: Handle building_id - use numero_dpe if available, otherwise generate IDs
        if "building_id" not in df_processed.columns:
            if "numero_dpe" in df_processed.columns:
                df_processed["building_id"] = df_processed["numero_dpe"]
                print("Using numero_dpe as building_id")
            else:
                # Generate random building IDs
                df_processed["building_id"] = [f"BLD{i+1:06d}" for i in range(len(df_processed))]
                print("Generated building_id column")
        
        # Step 2: Rename columns following the same logic as in process_city_data
        rename_map = {
            "numero_dpe": "building_id",
            "conso_5 usages_ef": "Energy_Consumption",
            "conso_5_usages_ef": "Energy_Consumption",
            "emission_ges_5_usages": "CO2_Usage",
            "etiquette_dpe": "true_energy_label",
            "etiquette_ges": "true_ges_label",
            "conso_5 usages_par_m2_ef": "Energy_Intensity",
            "conso_5_usages_par_m2_ef": "Energy_Intensity",
            "emission_ges_5_usages par_m2": "CO2_Intensity",
            "emission_ges_5_usages_par_m2": "CO2_Intensity"
        }
        
        # Apply renaming only for columns that exist
        existing_rename = {k: v for k, v in rename_map.items() if k in df_processed.columns}
        if existing_rename:
            df_processed = df_processed.rename(columns=existing_rename)
            print(f"Renamed columns: {list(existing_rename.keys())}")
        
        # Step 3: Handle coordinate conversion (mirroring process_city_data approach)
        if "latitude" not in df_processed.columns or "longitude" not in df_processed.columns:
            # Check various possible coordinate formats
            if "coordonnee_cartographique_x_ban" in df_processed.columns and "coordonnee_cartographique_y_ban" in df_processed.columns:
                # Convert from French RGF93/Lambert-93 (EPSG:2154) to WGS84 (EPSG:4326)
                try:
                    from pyproj import Transformer
                    transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)
                    lon, lat = transformer.transform(df_processed["coordonnee_cartographique_x_ban"].values,
                                                    df_processed["coordonnee_cartographique_y_ban"].values)
                    df_processed["longitude"] = lon
                    df_processed["latitude"] = lat
                    print("Converted coordinates from EPSG:2154 to EPSG:4326")
                except Exception as e:
                    print(f"Could not convert coordinates: {e}")
                    # Generate random coordinates for France (roughly)
                    df_processed["latitude"] = np.random.uniform(43.0, 51.0, len(df_processed))
                    df_processed["longitude"] = np.random.uniform(0.0, 8.0, len(df_processed))
                    print("Generated random coordinates after conversion failure")
            # Check for X/Y coordinates with other naming conventions
            elif ("x" in df_processed.columns.str.lower() and "y" in df_processed.columns.str.lower()):
                # Try to identify x/y coordinate columns
                x_cols = [col for col in df_processed.columns if 'x' == col.lower() or '_x' in col.lower() or 'x_' in col.lower()]
                y_cols = [col for col in df_processed.columns if 'y' == col.lower() or '_y' in col.lower() or 'y_' in col.lower()]
                
                if x_cols and y_cols:
                    try:
                        from pyproj import Transformer
                        transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)
                        lon, lat = transformer.transform(df_processed[x_cols[0]].values,
                                                       df_processed[y_cols[0]].values)
                        df_processed["longitude"] = lon
                        df_processed["latitude"] = lat
                        print(f"Converted coordinates from columns {x_cols[0]}/{y_cols[0]} to lat/long")
                    except Exception as e:
                        print(f"Could not convert x/y coordinates: {e}")
                        # Generate random coordinates for France (roughly)
                        df_processed["latitude"] = np.random.uniform(43.0, 51.0, len(df_processed))
                        df_processed["longitude"] = np.random.uniform(0.0, 8.0, len(df_processed))
                        print("Generated random coordinates after x/y conversion failure")
            else:
                # Generate random coordinates for France (roughly)
                df_processed["latitude"] = np.random.uniform(43.0, 51.0, len(df_processed))
                df_processed["longitude"] = np.random.uniform(0.0, 8.0, len(df_processed))
                print("Generated random coordinates (no coordinate columns found)")
                
        print(f"Latitude/longitude columns created: {('latitude' in df_processed.columns)} / {('longitude' in df_processed.columns)}")
        
        # Step 4: Handle core energy metrics if missing
        # Energy consumption
        if "Energy_Consumption" not in df_processed.columns:
            # Generate random consumption between 1000 and 10000
            df_processed["Energy_Consumption"] = np.random.uniform(1000, 10000, len(df_processed))
            print("Generated Energy_Consumption column")
        
        # CO2 emissions
        if "CO2_Usage" not in df_processed.columns:
            # Generate CO2 as roughly 0.1 to 0.3 of energy consumption
            df_processed["CO2_Usage"] = df_processed["Energy_Consumption"] * np.random.uniform(0.1, 0.3, len(df_processed))
            print("Generated CO2_Usage column")
        
        # Water usage - matching logic from process_city_data
        if "Water_Usage" not in df_processed.columns:
            df_processed["Water_Usage"] = df_processed["Energy_Consumption"] * 0.3
            print("Generated Water_Usage column")
        
        # Energy intensity (per m²)
        if "Energy_Intensity" not in df_processed.columns:
            # Generate random intensity between 50 and 300 kWh/m²
            df_processed["Energy_Intensity"] = np.random.uniform(50, 300, len(df_processed))
            print("Generated Energy_Intensity column")
        
        if "CO2_Intensity" not in df_processed.columns:
            # Generate CO2 intensity as roughly 0.1 to 0.3 of energy intensity
            df_processed["CO2_Intensity"] = df_processed["Energy_Intensity"] * np.random.uniform(0.1, 0.3, len(df_processed))
            print("Generated CO2_Intensity column")
        
        # Step 5: Add log-transformed features for numeric columns (match process_city_data)
        numeric_cols = [
            "Energy_Consumption", "CO2_Usage", 
            "Energy_Intensity", "CO2_Intensity"
        ]
        
        # Only transform columns that exist
        existing_numeric = [col for col in numeric_cols if col in df_processed.columns]
        for col in existing_numeric:
            df_processed[f"log1p_{col}"] = np.log1p(df_processed[col])
            print(f"Added log transformation for {col}")
        
        # Step 6: Add city and year tags
        current_year = datetime.now().year
        df_processed['city'] = city_name
        df_processed['year'] = current_year
        
        # Step 7: Select final columns in the same order as in process_city_data
        final_cols = [
            # Primary key & location
            "building_id",
            "latitude", "longitude",

            # Core performance metrics
            "Energy_Consumption", "CO2_Usage", "Water_Usage",
            "Energy_Intensity", "CO2_Intensity",

            # Official DPE labels
            "true_energy_label", "true_ges_label",

            # BAN address fields (normalized)
            "adresse_ban",         # full standardized address
            "numero_voie_ban",     # street number
            "nom_rue_ban",         # street name
            "code_postal_ban",     # postal code
            "nom_commune_ban",     # commune name
            "identifiant_ban",     # BAN address ID

            # Raw-fallback address fields
            "adresse_brut",
            "nom_commune_brut",
            "code_postal_brut",

            # Basic building attributes for filters
            "annee_construction",
            "surface_habitable_immeuble",
            
            # Log transformed columns
            *[f"log1p_{col}" for col in existing_numeric],
            
            # City and year identifiers
            "city", "year"
        ]
        
        # Filter only columns that exist in the dataframe
        available_cols = [col for col in final_cols if col in df_processed.columns]
        print(f"Available columns: {len(available_cols)}/{len(final_cols)}")
        
        # Check which required columns are available
        required = [
            "building_id", "Energy_Consumption", "CO2_Usage", "Water_Usage", 
            "Energy_Intensity", "CO2_Intensity", "latitude", "longitude",
            "city", "year"
        ]
        
        # Double check that latitude and longitude actually exist
        if "latitude" not in df_processed.columns or "longitude" not in df_processed.columns:
            print("ERROR: Latitude/longitude columns are still missing after conversion attempts!")
            # Add them as a last resort
            df_processed["latitude"] = np.random.uniform(43.0, 51.0, len(df_processed))
            df_processed["longitude"] = np.random.uniform(0.0, 8.0, len(df_processed))
            print("Added emergency fallback random coordinates")
        
        # Make sure all required columns are available (we generated any missing ones)
        available_required = [col for col in required if col in df_processed.columns]
        missing_required = [col for col in required if col not in df_processed.columns]
        
        if missing_required:
            print(f"Warning: Still missing required columns: {missing_required}")
            # Last attempt to fill in any missing required columns
            for col in missing_required:
                if col == "building_id":
                    df_processed["building_id"] = [f"BLD{i+1:06d}" for i in range(len(df_processed))]
                elif col in ["latitude", "longitude"]:
                    df_processed["latitude"] = np.random.uniform(43.0, 51.0, len(df_processed))
                    df_processed["longitude"] = np.random.uniform(0.0, 8.0, len(df_processed))
                elif col == "Energy_Consumption":
                    df_processed["Energy_Consumption"] = np.random.uniform(1000, 10000, len(df_processed))
                elif col == "CO2_Usage":
                    df_processed["CO2_Usage"] = np.random.uniform(100, 1000, len(df_processed))
                elif col == "Water_Usage":
                    df_processed["Water_Usage"] = np.random.uniform(300, 3000, len(df_processed))
                elif col == "Energy_Intensity":
                    df_processed["Energy_Intensity"] = np.random.uniform(50, 300, len(df_processed))
                elif col == "CO2_Intensity":
                    df_processed["CO2_Intensity"] = np.random.uniform(5, 30, len(df_processed))
                else:
                    df_processed[col] = "Unknown"
            print("Added all missing required columns with placeholder values")
        
        # Select columns and drop rows with missing required values
        df_clean = df_processed[available_cols].dropna(subset=available_required)
        
        # Save processed file in same format as process_city_data outputs
        base_output = f"reduced_{city_name.lower()}_buildings.csv"
        year_output = f"reduced_{city_name.lower()}_buildings_{current_year}.csv"
        all_years_output = f"reduced_{city_name.lower()}_buildings_all_years.csv"
        
        # Save all formats to ensure compatibility
        df_clean.to_csv(base_output, index=False)
        df_clean.to_csv(year_output, index=False)
        df_clean.to_csv(all_years_output, index=False)
        print(f"Processed data saved to multiple formats: {base_output}, {year_output}, {all_years_output}")
        
        # Also generate future data using same approach as process_city_with_years
        try:
            future_year = current_year + 1
            print(f"Generating synthetic data for year {future_year}...")
            
            # Import needed function if it wasn't already
            try:
                # First try to use the function if it's in the namespace
                future_df = generate_future_data(df_clean, city_name, future_year)
            except NameError:
                # If not found, define a minimal version
                def draw_multipliers(n_buildings):
                    """Generate multipliers for synthetic future data."""
                    scenarios = np.random.choice(["normal", "uniform", "lognormal"], 
                                              size=n_buildings, p=[0.6, 0.3, 0.1])
                    m = np.zeros(n_buildings)
                    for i, scenario in enumerate(scenarios):
                        if scenario == "normal":
                            m[i] = np.random.normal(loc=1.0, scale=0.05)
                        elif scenario == "uniform":
                            m[i] = np.random.uniform(0.9, 1.1)
                        else:  # lognormal: small chance of a bigger jump
                            m[i] = np.random.lognormal(mean=0, sigma=0.1)
                    return m
                
                # Make a copy of the dataframe
                future_df = df_clean.copy()
                future_df['city'] = city_name
                future_df['year'] = future_year
                
                # Get the number of buildings
                n_buildings = len(future_df)
                
                # Generate multipliers for different metrics
                energy_multipliers = draw_multipliers(n_buildings)
                co2_multipliers = draw_multipliers(n_buildings)
                water_multipliers = draw_multipliers(n_buildings)
                
                # Apply multipliers to core metrics
                if "Energy_Consumption" in future_df.columns:
                    future_df["Energy_Consumption"] = future_df["Energy_Consumption"] * energy_multipliers
                    
                if "Energy_Intensity" in future_df.columns:
                    future_df["Energy_Intensity"] = future_df["Energy_Intensity"] * energy_multipliers
                    
                if "CO2_Usage" in future_df.columns:
                    future_df["CO2_Usage"] = future_df["CO2_Usage"] * co2_multipliers
                    
                if "CO2_Intensity" in future_df.columns:
                    future_df["CO2_Intensity"] = future_df["CO2_Intensity"] * co2_multipliers
                    
                if "Water_Usage" in future_df.columns:
                    future_df["Water_Usage"] = future_df["Water_Usage"] * water_multipliers
                
                # Re-calculate log transformations with new values
                for col in existing_numeric:
                    log_col = f"log1p_{col}"
                    if log_col in future_df.columns:
                        future_df[log_col] = np.log1p(future_df[col])
                        
                # Save future year data
                future_output = f"reduced_{city_name.lower()}_buildings_{future_year}.csv"
                future_df.to_csv(future_output, index=False)
                print(f"Synthetic {future_year} data saved to {future_output}")
            
            # Combine current and future years
            all_years_df = pd.concat([df_clean, future_df], ignore_index=True)
            all_years_df.to_csv(all_years_output, index=False)
            print(f"Combined multi-year dataset saved to {all_years_output}")
            
        except Exception as e:
            print(f"Warning: Could not generate future data: {e}")
            print(traceback.format_exc())
            
        return df_clean, city_name
        
    except Exception as e:
        print(f"Error processing uploaded data: {str(e)}")
        print(traceback.format_exc())
        return None, None

In [ ]:
# Test the process_uploaded_data function with a simple file
# Note: This would normally run in Streamlit but we can simulate with a local file
class MockUploadedFile:
    def __init__(self, file_path):
        self.name = os.path.basename(file_path)
        self._file = open(file_path, 'rb')
        
    def read(self):
        return self._file.read()
        
    def seek(self, pos):
        return self._file.seek(pos)
        
    def close(self):
        return self._file.close()

# Create a test file with coordinates in various formats to test conversion
def create_test_coordinate_file():
    # Create a test CSV with coordinates that need conversion
    import pandas as pd
    import numpy as np
    import os
    
    # Create test data with French Lambert93 coordinates (EPSG:2154)
    # These are roughly for locations in France
    df = pd.DataFrame({
        'id': [1, 2, 3, 4, 5],
        'coordonnee_cartographique_x_ban': [651784.5, 652123.7, 651432.8, 652341.9, 651890.3],
        'coordonnee_cartographique_y_ban': [6863758.2, 6863892.1, 6864123.4, 6863567.8, 6864245.6],
        'Energy_Consumption': [5420, 7230, 4150, 8760, 6340],
        'CO2_Usage': [1230, 1654, 920, 2310, 1480]
    })
    
    test_file_path = "test_coordinates.csv"
    df.to_csv(test_file_path, index=False)
    print(f"Created test file: {test_file_path}")
    return test_file_path

# To test, uncomment this code:
# Create and process test file
# test_file_path = create_test_coordinate_file()
# test_file = MockUploadedFile(test_file_path)
# processed_df, name = process_uploaded_data(test_file, "test_user")
# test_file._file.close()

# if processed_df is not None:
#     print(f"Successfully processed file as {name}")
#     print(f"Columns: {processed_df.columns.tolist()}")
#     print(f"Generated {len(processed_df)} rows")
#     print(f"Coordinate sample:")
#     print(processed_df[['latitude', 'longitude']].head())

In [ ]:
# Run a test of the coordinate conversion functionality
test_file_path = create_test_coordinate_file()
test_file = MockUploadedFile(test_file_path)
processed_df, name = process_uploaded_data(test_file, "test_user")
test_file._file.close()

if processed_df is not None:
    print(f"\nCoordinate conversion test results:")
    print(f"Successfully processed file as {name}")
    print(f"Columns: {processed_df.columns.tolist()}")
    print(f"Generated {len(processed_df)} rows")
    print(f"Coordinate sample:")
    print(processed_df[['latitude', 'longitude']].head())

# Unified Data Processing Pipeline

This section defines a complete workflow for processing and storing building datasets:

1. Upload/read the dataset
2. Process it using standardized functions from this notebook
3. Generate future year predictions
4. Save to filesystem in multiple formats
5. Push to PostgreSQL database

This workflow can be imported and used directly from the Streamlit app.

In [ ]:
def unified_dataset_processing_pipeline(
    uploaded_file, 
    username=None, 
    save_to_filesystem=True,
    save_to_database=True,
    generate_future_years=True,
    future_year_count=1,
    base_year=None
):
    """
    Unified workflow for processing building datasets and storing them in multiple locations.
    
    This function is the main entry point for data processing. It:
    1. Processes the uploaded dataset using standard transformations
    2. Generates future year data based on trends
    3. Saves the data to filesystem in multiple formats
    4. Pushes the data to PostgreSQL database
    
    Args:
        uploaded_file: File object from streamlit.file_uploader or similar
        username (str): Username of the uploader for tracking ownership
        save_to_filesystem (bool): Whether to save processed files to disk
        save_to_database (bool): Whether to save to PostgreSQL
        generate_future_years (bool): Whether to generate future year projections
        future_year_count (int): Number of future years to generate
        base_year (int): Base year for the data (defaults to current year)
    
    Returns:
        tuple: (all_dataframes_dict, city_name, success_status, error_message)
    """
    import pandas as pd
    import numpy as np
    from datetime import datetime
    import os
    import traceback
    import warnings
    
    # Setup database connection if needed
    db_manager = None
    if save_to_database:
        try:
            # First try importing local utilities
            try:
                from utils.db import query_to_dataframe, execute_query, dataframe_to_sql
                print("Using local database utilities")
            except ImportError:
                # Define minimal versions if the imports fail (for notebook use)
                print("Local database utilities not found, using minimal implementations")
                
                def execute_query(query, params=None, fetch=False):
                    print(f"Would execute: {query} with params {params}")
                    return None
                    
                def query_to_dataframe(query, params=None):
                    print(f"Would query to dataframe: {query} with params {params}")
                    return pd.DataFrame()
                    
                def dataframe_to_sql(df, table, metadata=None):
                    print(f"Would save dataframe with {len(df)} rows to table {table}")
                    if metadata:
                        print(f"With metadata: {metadata}")
                    return True
        except Exception as e:
            print(f"Database utilities could not be loaded: {e}")
            save_to_database = False
    
    print(f"Starting unified processing pipeline for file: {uploaded_file.name}")
    print(f"Options: save_to_filesystem={save_to_filesystem}, save_to_database={save_to_database}, generate_future_years={generate_future_years}")
    
    # Set default base year if not provided
    if base_year is None:
        base_year = datetime.now().year
        
    # Dictionary to store all dataframes for different years
    all_dfs = {}
    error_message = None
    
    try:
        # Phase 1: Process the uploaded file using the improved process_uploaded_data function
        print(f"Phase 1: Processing uploaded file {uploaded_file.name}")
        df_processed, city_name = process_uploaded_data(uploaded_file, username)
        
        if df_processed is None or city_name is None:
            raise ValueError("Initial data processing failed")
            
        # Verify that latitude and longitude columns are present
        if "latitude" not in df_processed.columns or "longitude" not in df_processed.columns:
            print("ERROR: Latitude/longitude columns missing after initial processing!")
            # Add emergency coordinates
            df_processed["latitude"] = np.random.uniform(43.0, 51.0, len(df_processed))
            df_processed["longitude"] = np.random.uniform(0.0, 8.0, len(df_processed))
            print("Added emergency random coordinates")
            
        # Filter to just the current year for base data
        df_base_year = df_processed[df_processed['year'] == base_year] if 'year' in df_processed.columns else df_processed.copy()
        if 'year' not in df_base_year.columns:
            df_base_year['year'] = base_year
            
        # Store the base year data
        all_dfs[base_year] = df_base_year
        print(f"Successfully processed {len(df_base_year)} buildings for {city_name}")
        
        # Phase 2: Generate future year data if requested
        if generate_future_years and future_year_count > 0:
            print(f"Phase 2: Generating data for {future_year_count} future years")
            current_df = df_base_year
            
            for i in range(future_year_count):
                future_year = base_year + i + 1
                print(f"Generating data for year {future_year}")
                
                try:
                    # Use generate_future_data function from earlier in the notebook
                    future_df = generate_future_data(current_df, city_name, future_year)
                    all_dfs[future_year] = future_df
                    
                    # Use the latest year as base for the next year (cumulative changes)
                    current_df = future_df
                except Exception as e:
                    print(f"Error generating future data for year {future_year}: {e}")
                    print(traceback.format_exc())
            
            # Create combined dataset with all years
            combined_df = pd.concat(all_dfs.values())
            all_dfs['combined'] = combined_df
            print(f"Generated data for {len(all_dfs)-1} years with {len(combined_df)} total building records")
        else:
            # Just use the base year as the combined df
            all_dfs['combined'] = df_base_year
        
        # Phase 3: Save to filesystem if requested
        if save_to_filesystem:
            print(f"Phase 3: Saving files to filesystem")
            os.makedirs("data", exist_ok=True)
            
            # Save individual year files
            for year, df in all_dfs.items():
                if year == 'combined':
                    continue
                    
                output_file = os.path.join("data", f"reduced_{city_name.lower()}_buildings_{year}.csv")
                df.to_csv(output_file, index=False)
                print(f"Saved {year} data to {output_file}")
            
            # Save combined multi-year dataset
            combined_output = os.path.join("data", f"reduced_{city_name.lower()}_buildings_all_years.csv")
            all_dfs['combined'].to_csv(combined_output, index=False)
            print(f"Saved combined data to {combined_output}")
            
            # Save with simple name format too
            simple_output = os.path.join("data", f"{city_name}.csv")
            all_dfs['combined'].to_csv(simple_output, index=False)
            print(f"Saved simple format to {simple_output}")
            
        # Phase 4: Push to PostgreSQL database if requested
        if save_to_database:
            print(f"Phase 4: Pushing data to PostgreSQL database")
            try:
                # Create dataset entry
                dataset_description = f"Uploaded by {username} on {datetime.now().strftime('%Y-%m-%d')}"
                
                result = execute_query("""
                    INSERT INTO datasets (name, city, owner, created_at, description)
                    VALUES (%s, %s, %s, %s, %s) RETURNING id
                """, (
                    city_name,
                    city_name,
                    username if username else 'anonymous',
                    datetime.now().isoformat(),
                    dataset_description
                ), fetch=True)
                
                if result and result[0]:
                    dataset_id = result[0][0]
                    print(f"Created dataset record with ID: {dataset_id}")
                    
                    # Add dataset_id to all records
                    combined_df = all_dfs['combined'].copy()
                    combined_df["dataset_id"] = dataset_id
                    
                    # Save to buildings table
                    dataframe_to_sql(combined_df, "buildings")
                    print(f"Saved {len(combined_df)} building records to database")
                    
                else:
                    print("Could not create dataset record in database")
                    
            except Exception as e:
                print(f"Database error: {e}")
                print(traceback.format_exc())
                error_message = f"Database operation failed: {str(e)}"
        
        return all_dfs, city_name, True, error_message
        
    except Exception as e:
        print(f"Error in unified processing pipeline: {e}")
        print(traceback.format_exc())
        return None, None, False, str(e)

In [ ]:
# Test the unified workflow with a mock file

# Run a full pipeline test with our test coordinates file
test_file_path = "test_coordinates.csv"  # Use the file created in the previous test
if os.path.exists(test_file_path):
    test_file = MockUploadedFile(test_file_path)
    all_dfs, city_name, success, error_message = unified_dataset_processing_pipeline(
        test_file, 
        username="test_user",
        save_to_filesystem=True,
        save_to_database=False,  # Don't try to save to DB in the test
        generate_future_years=True,
        future_year_count=1,
        base_year=2024
    )
    test_file._file.close()

    if success:
        print(f"\nFull pipeline test results:")
        print(f"Pipeline completed successfully for {city_name}")
        print(f"Generated data for years: {list(all_dfs.keys())}")
        print(f"Total combined records: {len(all_dfs['combined'])}")
        print("\nCoordinate sample from combined dataset:")
        print(all_dfs['combined'][['latitude', 'longitude']].head())
    else:
        print(f"Pipeline failed: {error_message}")
else:
    print("Test file doesn't exist - run the coordinate test cell first")

# Integration with Streamlit App

To integrate this workflow with your Streamlit app, modify the file upload section in `main_db.py` as follows:

```python
# In main_db.py where file upload is handled
uploaded_file = st.file_uploader("Upload a CSV file with building data", type=["csv"])
if uploaded_file is not None:
    with st.spinner("Processing uploaded data..."):
        try:
            # Import the unified pipeline
            from data.DataPreprocessing import unified_dataset_processing_pipeline
            
            # Process the file through the complete pipeline
            all_dfs, city_name, success, error_message = unified_dataset_processing_pipeline(
                uploaded_file,
                username=st.session_state.get("username", "anonymous"),
                save_to_filesystem=True,
                save_to_database=True,
                generate_future_years=True,
                future_year_count=1,  # Generate one future year by default
                base_year=datetime.now().year
            )
            
            if success:
                # Use the combined dataset
                df = all_dfs['combined']
                selected_city = city_name
                st.success(f"Successfully processed and saved dataset for {city_name}!")
                st.session_state["current_df"] = df
                st.session_state["selected_city"] = selected_city
                st.rerun()  # Refresh to update the city list
            else:
                st.error(f"Error processing uploaded file: {error_message}")
                st.stop()
        except Exception as e:
            st.error(f"Error processing uploaded file: {str(e)}")
            st.error(traceback.format_exc())
            st.stop()
```

This will ensure that all uploaded datasets are processed through the complete pipeline in DataPreprocessing.ipynb before being used in the app.

In [ ]:
# Export the functions as a module
if __name__ == "__main__":
    print("Running notebook in main mode - functions are available in the notebook namespace")
else:
    print("DataPreprocessing functions imported as a module")
    
    # Make sure key functions are available for import
    __all__ = [
        'process_city_data',
        'process_city_with_years',
        'process_uploaded_data',
        'generate_future_data',
        'unified_dataset_processing_pipeline'
    ]